# Phase 2 - Model Comparison & Evaluation

This notebook compares all Phase 2 models trained on the augmented dataset:
- SSD Lite 320 MobileNetV3
- DETR-ResNet-50
- Faster R-CNN ResNet-50 FPN
- YOLOv8

**Comparison Metrics:**
- mAP@50 and mAP@50-95
- FPS (inference speed)
- Training loss curves
- Visual comparison on same images
- Final recommendations

## 1. Imports & Path Setup

In [2]:
import os
from pathlib import Path
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torchvision.transforms.functional import convert_image_dtype

from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    fasterrcnn_resnet50_fpn,
)
from transformers import DetrForObjectDetection, DetrImageProcessor
from ultralytics import YOLO

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == "Phase 2" else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Dataset paths
DATASET_ROOT = BASE_DIR / "data" / "processed" / "parking_dataset_augmented"
IMAGES_DIR = DATASET_ROOT / "images"
ANNOTATIONS_DIR = DATASET_ROOT / "annotations"

# Checkpoint paths
CHECKPOINT_DIRS = {
    "ssd": BASE_DIR / "checkpoints" / "phase2_ssd_parking",
    "detr": BASE_DIR / "checkpoints" / "phase2_detr_parking",
    "frcnn": BASE_DIR / "checkpoints" / "phase2_faster_rcnn_parking",
    "yolo": BASE_DIR / "runs" / "yolo_parking_phase2",
}

class_names = ["free_parking_space", "not_free_parking_space"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'numpy'

## 2. Load All Trained Models

In [ ]:
# Load all Phase 2 trained models from their checkpoints

models_loaded = {}

# Load SSD Phase 2
ssd_ckpt_p2 = CHECKPOINT_DIRS['ssd'] / "best_model.pt"
if ssd_ckpt_p2.exists():
    try:
        ssd_model_p2 = ssdlite320_mobilenet_v3_large(num_classes=3).to(device)  # 2 classes + bg
        checkpoint = torch.load(ssd_ckpt_p2, map_location=device)
        ssd_model_p2.load_state_dict(checkpoint['model_state_dict'])
        ssd_model_p2.eval()
        models_loaded['SSD'] = ssd_model_p2
        print("✓ SSD Phase 2 model loaded")
    except Exception as e:
        print(f"⚠ Error loading SSD: {e}")

# Load DETR Phase 2
detr_ckpt_p2 = CHECKPOINT_DIRS['detr'] / "best_model.pt"
if detr_ckpt_p2.exists():
    try:
        from transformers import DetrConfig
        config = DetrConfig.from_pretrained("facebook/detr-resnet-50")
        config.num_labels = 2
        detr_model_p2 = DetrForObjectDetection.from_pretrained(
            "facebook/detr-resnet-50", config=config, ignore_mismatched_sizes=True
        ).to(device)
        checkpoint = torch.load(detr_ckpt_p2, map_location=device)
        detr_model_p2.load_state_dict(checkpoint['model_state_dict'])
        detr_model_p2.eval()
        models_loaded['DETR'] = detr_model_p2
        print("✓ DETR Phase 2 model loaded")
    except Exception as e:
        print(f"⚠ Error loading DETR: {e}")

# Load Faster R-CNN Phase 2
frcnn_ckpt_p2 = CHECKPOINT_DIRS['frcnn'] / "best_model.pt"
if frcnn_ckpt_p2.exists():
    try:
        frcnn_model_p2 = fasterrcnn_resnet50_fpn(num_classes=3).to(device)  # 2 classes + bg
        checkpoint = torch.load(frcnn_ckpt_p2, map_location=device)
        frcnn_model_p2.load_state_dict(checkpoint['model_state_dict'])
        frcnn_model_p2.eval()
        models_loaded['Faster R-CNN'] = frcnn_model_p2
        print("✓ Faster R-CNN Phase 2 model loaded")
    except Exception as e:
        print(f"⚠ Error loading Faster R-CNN: {e}")

# Load YOLO Phase 2
yolo_best_p2 = CHECKPOINT_DIRS['yolo'] / "weights" / "best.pt"
if not yolo_best_p2.exists():
    yolo_best_p2 = list(CHECKPOINT_DIRS['yolo'].rglob("best.pt"))
    if yolo_best_p2:
        yolo_best_p2 = yolo_best_p2[0]

if yolo_best_p2.exists() if isinstance(yolo_best_p2, Path) else False:
    try:
        yolo_model_p2 = YOLO(str(yolo_best_p2))
        models_loaded['YOLO'] = yolo_model_p2
        print("✓ YOLO Phase 2 model loaded")
    except Exception as e:
        print(f"⚠ Error loading YOLO: {e}")

print(f"\nTotal Phase 2 models loaded: {len(models_loaded)}")
for name in models_loaded.keys():
    print(f"  - {name}")

## 3. Dataset Setup for Evaluation

In [ ]:
# Create JSON dataset for evaluation (same as training notebooks)
from sklearn.model_selection import train_test_split

class JsonDetectionDataset(Dataset):
    """Dataset for loading JSON annotations"""
    def __init__(self, annotation_files, images_dir):
        self.annotation_files = annotation_files
        self.images_dir = images_dir
    
    def __len__(self):
        return len(self.annotation_files)
    
    def __getitem__(self, idx):
        ann_path = self.annotation_files[idx]
        with open(ann_path, 'r') as f:
            ann = json.load(f)
        
        img_name = ann.get('image_name', ann_path.stem + '.jpg')
        img_path = self.images_dir / img_name
        img = read_image(str(img_path))
        
        C, H, W = img.shape
        if C == 1:
            img = img.repeat(3, 1, 1)
        elif C == 4:
            img = img[:3, :, :]
        
        img = convert_image_dtype(img, dtype=torch.float32)
        _, H, W = img.shape
        
        boxes = []
        labels = []
        for space in ann.get('spaces', []):
            bbox = space.get('bbox', [])
            occupied = space.get('occupied', False)
            if len(bbox) == 4:
                x_min, y_min, x_max, y_max = map(float, bbox)
                x_min = max(0.0, min(float(W), x_min))
                y_min = max(0.0, min(float(H), y_min))
                x_max = max(0.0, min(float(W), x_max))
                y_max = max(0.0, min(float(H), y_max))
                if x_max > x_min and y_max > y_min:
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(2 if occupied else 1)  # For torchvision models: 1=free, 2=occupied
        
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        
        return img, {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([idx])}

# Get annotation files
annotation_files = sorted(list(ANNOTATIONS_DIR.glob("*.json")))
val_files = annotation_files[-len(annotation_files)//5:]  # Use last 20% for evaluation

print(f"Validation files for evaluation: {len(val_files):,}")

# Create validation dataset
val_dataset_eval = JsonDetectionDataset(val_files, IMAGES_DIR)
print(f"Validation dataset created: {len(val_dataset_eval)} samples")

## 4. Comprehensive Model Evaluation

In [ ]:
# Try to import torchmetrics for mAP evaluation
try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    TORCHMETRICS_AVAILABLE = True
    MeanAveragePrecision = None

# Store evaluation results
evaluation_results = {}

In [ ]:
# Evaluate SSD
if 'SSD' in models_loaded:
    print("\nEvaluating SSD...")
    if TORCHMETRICS_AVAILABLE:
        metric = MeanAveragePrecision()
        model = models_loaded['SSD']
        model.eval()
        
        for i in range(min(100, len(val_dataset_eval))):
            img, tgt = val_dataset_eval[i]
            img_gpu = img.to(device).unsqueeze(0)
            
            with torch.no_grad():
                pred = model([img_gpu.squeeze(0)])[0]
            
            preds = [{'boxes': pred['boxes'].cpu(), 'scores': pred['scores'].cpu(), 'labels': pred['labels'].cpu()}]
            targets = [{'boxes': tgt['boxes'], 'labels': tgt['labels']}]
            metric.update(preds, targets)
        
        ssd_metrics = metric.compute()
        evaluation_results['SSD'] = {
            'mAP50': float(ssd_metrics['map_50'].item()) if 'map_50' in ssd_metrics else 0.0,
            'mAP50_95': float(ssd_metrics['map'].item()) if 'map' in ssd_metrics else 0.0,
        }
        print(f"  mAP@50: {evaluation_results['SSD']['mAP50']:.4f}")
        print(f"  mAP@50-95: {evaluation_results['SSD']['mAP50_95']:.4f}")

# Evaluate Faster R-CNN
if 'Faster R-CNN' in models_loaded:
    print("\nEvaluating Faster R-CNN...")
    if TORCHMETRICS_AVAILABLE:
        metric = MeanAveragePrecision()
        model = models_loaded['Faster R-CNN']
        model.eval()
        
        for i in range(min(100, len(val_dataset_eval))):
            img, tgt = val_dataset_eval[i]
            img_gpu = img.to(device).unsqueeze(0)
            
            with torch.no_grad():
                pred = model([img_gpu.squeeze(0)])[0]
            
            preds = [{'boxes': pred['boxes'].cpu(), 'scores': pred['scores'].cpu(), 'labels': pred['labels'].cpu()}]
            targets = [{'boxes': tgt['boxes'], 'labels': tgt['labels']}]
            metric.update(preds, targets)
        
        frcnn_metrics = metric.compute()
        evaluation_results['Faster R-CNN'] = {
            'mAP50': float(frcnn_metrics['map_50'].item()) if 'map_50' in frcnn_metrics else 0.0,
            'mAP50_95': float(frcnn_metrics['map'].item()) if 'map' in frcnn_metrics else 0.0,
        }
        print(f"  mAP@50: {evaluation_results['Faster R-CNN']['mAP50']:.4f}")
        print(f"  mAP@50-95: {evaluation_results['Faster R-CNN']['mAP50_95']:.4f}")

# Evaluate DETR (requires processor)
if 'DETR' in models_loaded:
    print("\nEvaluating DETR...")
    if TORCHMETRICS_AVAILABLE:
        try:
            processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
            metric = MeanAveragePrecision()
            model = models_loaded['DETR']
            model.eval()
            
            for i in range(min(100, len(val_dataset_eval))):
                img, tgt = val_dataset_eval[i]
                inputs = processor(images=img.cpu(), return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                target_sizes = torch.tensor([img.shape[-2:]]).to(device)
                
                with torch.no_grad():
                    outputs = model(**inputs)
                    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.0)[0]
                
                preds = [{'boxes': results['boxes'].cpu(), 'scores': results['scores'].cpu(), 'labels': results['labels'].cpu()}]
                targets = [{'boxes': tgt['boxes'], 'labels': tgt['labels']}]
                metric.update(preds, targets)
            
            detr_metrics = metric.compute()
            evaluation_results['DETR'] = {
                'mAP50': float(detr_metrics['map_50'].item()) if 'map_50' in detr_metrics else 0.0,
                'mAP50_95': float(detr_metrics['map'].item()) if 'map' in detr_metrics else 0.0,
            }
            print(f"  mAP@50: {evaluation_results['DETR']['mAP50']:.4f}")
            print(f"  mAP@50-95: {evaluation_results['DETR']['mAP50_95']:.4f}")
        except Exception as e:
            print(f"  Error evaluating DETR: {e}")

# Evaluate YOLO (using built-in validation)
if 'YOLO' in models_loaded:
    print("\nEvaluating YOLO...")
    try:
        # Create YOLO dataset path (if converted)
        yolo_data_yaml = BASE_DIR / "data" / "processed" / "yolo_parking_phase2" / "data.yaml"
        if yolo_data_yaml.exists():
            metrics = models_loaded['YOLO'].val(data=str(yolo_data_yaml))
            evaluation_results['YOLO'] = {
                'mAP50': float(metrics.box.map50) if hasattr(metrics.box, 'map50') else 0.0,
                'mAP50_95': float(metrics.box.map) if hasattr(metrics.box, 'map') else 0.0,
            }
            print(f"  mAP@50: {evaluation_results['YOLO']['mAP50']:.4f}")
    except Exception as e:
        print(f"  Error evaluating YOLO: {e}")

## 5. FPS Benchmarking

In [ ]:
# Measure inference speed (FPS) for all models
fps_results = {}

@torch.no_grad()
def measure_fps_torchvision(model, dataset, num_samples=30):
    model.eval()
    t0 = time.time()
    for i in range(num_samples):
        img, _ = dataset[i % len(dataset)]
        img_gpu = img.to(device).unsqueeze(0)
        _ = model([img_gpu.squeeze(0)])
    t1 = time.time()
    return num_samples / (t1 - t0)

# Benchmark SSD
if 'SSD' in models_loaded:
    fps_results['SSD'] = measure_fps_torchvision(models_loaded['SSD'], val_dataset_eval, num_samples=30)
    print(f"SSD FPS: {fps_results['SSD']:.2f} frames/s")

# Benchmark Faster R-CNN
if 'Faster R-CNN' in models_loaded:
    fps_results['Faster R-CNN'] = measure_fps_torchvision(models_loaded['Faster R-CNN'], val_dataset_eval, num_samples=30)
    print(f"Faster R-CNN FPS: {fps_results['Faster R-CNN']:.2f} frames/s")

# Benchmark DETR
if 'DETR' in models_loaded:
    try:
        processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
        model = models_loaded['DETR']
        model.eval()
        t0 = time.time()
        for i in range(30):
            img, _ = val_dataset_eval[i % len(val_dataset_eval)]
            inputs = processor(images=img.cpu(), return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            _ = model(**inputs)
        t1 = time.time()
        fps_results['DETR'] = 30 / (t1 - t0)
        print(f"DETR FPS: {fps_results['DETR']:.2f} frames/s")
    except Exception as e:
        print(f"Error benchmarking DETR: {e}")

# Benchmark YOLO
if 'YOLO' in models_loaded:
    try:
        val_images = val_files[:30]
        img_paths = [IMAGES_DIR / (Path(f).stem + '.jpg') for f in val_images]
        img_paths = [p for p in img_paths if p.exists()][:30]
        
        if img_paths:
            model = models_loaded['YOLO']
            model.fuse()
            t0 = time.time()
            for img_path in img_paths:
                _ = model(str(img_path), verbose=False)
            t1 = time.time()
            fps_results['YOLO'] = len(img_paths) / (t1 - t0)
            print(f"YOLO FPS: {fps_results['YOLO']:.2f} frames/s")
    except Exception as e:
        print(f"Error benchmarking YOLO: {e}")

## 6. Comprehensive Comparison Table

In [ ]:
# Create comprehensive comparison table
comparison_data = {
    'Model': [],
    'mAP@50': [],
    'mAP@50-95': [],
    'FPS': [],
    'Model Type': [],
}

model_types = {
    'SSD': 'One-stage (anchor-based)',
    'Faster R-CNN': 'Two-stage',
    'DETR': 'Transformer-based',
    'YOLO': 'One-stage (anchor-free)',
}

for model_name in ['SSD', 'Faster R-CNN', 'DETR', 'YOLO']:
    if model_name in evaluation_results:
        comparison_data['Model'].append(model_name)
        comparison_data['mAP@50'].append(evaluation_results[model_name].get('mAP50', 0.0))
        comparison_data['mAP@50-95'].append(evaluation_results[model_name].get('mAP50_95', 0.0))
        comparison_data['FPS'].append(fps_results.get(model_name, 0.0))
        comparison_data['Model Type'].append(model_types.get(model_name, 'Unknown'))

comparison_df = pd.DataFrame(comparison_data)

if len(comparison_df) > 0:
    print("Comprehensive Model Comparison:")
    print("=" * 80)
    print(comparison_df.to_string(index=False))
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    models = comparison_df['Model'].tolist()
    
    # mAP@50 comparison
    axes[0].bar(models, comparison_df['mAP@50'], color=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'][:len(models)])
    axes[0].set_ylabel('mAP@50')
    axes[0].set_title('mAP@50 Comparison')
    axes[0].set_ylim(0, 1)
    axes[0].grid(True, alpha=0.3)
    
    # mAP@50-95 comparison
    axes[1].bar(models, comparison_df['mAP@50-95'], color=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'][:len(models)])
    axes[1].set_ylabel('mAP@50-95')
    axes[1].set_title('mAP@50-95 Comparison')
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)
    
    # FPS comparison
    fps_values = [float(str(f).replace(' frames/s', '')) if isinstance(f, str) else f for f in comparison_df['FPS']]
    axes[2].bar(models, fps_values, color=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'][:len(models)])
    axes[2].set_ylabel('FPS')
    axes[2].set_title('Inference Speed (FPS) Comparison')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:


## 7. Phase 1 vs Phase 2 Comparison

In [ ]:
# Compare Phase 1 and Phase 2 performance for each model

phase_comparison_summary = {
    'Model': [],
    'Phase 1 mAP@50': [],
    'Phase 2 mAP@50': [],
    'Improvement': [],
    'Phase 1 mAP@50-95': [],
    'Phase 2 mAP@50-95': [],
    'Improvement (mAP@50-95)': [],
}

# Load Phase 1 metrics from checkpoints if available
phase1_checkpoints = {
    'SSD': BASE_DIR / "ssd_parking_best.pt",
    'Faster R-CNN': BASE_DIR / "faster_rcnn_parking_best.pt",
    'YOLO': BASE_DIR / "yolo_parking_best.pt",
}

print("Phase 1 vs Phase 2 Comparison:")
print("=" * 80)
print("(Note: Requires Phase 1 models to be trained and evaluated)")
print("  2. Evaluate Phase 1 models")
print("  3. Load Phase 1 metrics and compare with Phase 2 results above")

# This will be populated after Phase 1 evaluation
phase_comparison_df = pd.DataFrame(phase_comparison_summary)
if len(phase_comparison_df) > 0:


## 8. Benchmarking with Literature

In [ ]:
# Compare results with published baselines
# NOTE: Replace with actual literature values from research papers

literature_baselines = {
    'YOLOv5 (Parking)': {'mAP50': 0.75, 'mAP50_95': 0.52, 'FPS': 30, 'Reference': 'Example Paper 2020'},
    'Faster R-CNN (Parking)': {'mAP50': 0.82, 'mAP50_95': 0.65, 'FPS': 8, 'Reference': 'Example Paper 2021'},
    'SSD (Parking)': {'mAP50': 0.70, 'mAP50_95': 0.48, 'FPS': 25, 'Reference': 'Example Paper 2019'},
    # Add more baselines from actual research papers
}

# Your results
your_results = {}
for model_name in ['SSD', 'Faster R-CNN', 'DETR', 'YOLO']:
    if model_name in evaluation_results:
        your_results[model_name] = {
            'mAP50': evaluation_results[model_name].get('mAP50', 0.0),
            'mAP50_95': evaluation_results[model_name].get('mAP50_95', 0.0),
            'FPS': fps_results.get(model_name, 0.0),
        }

# Create benchmarking comparison
benchmark_data = {
    'Model': [],
    'mAP@50': [],
    'mAP@50-95': [],
    'FPS': [],
    'Source': [],
}

for model_name, metrics in literature_baselines.items():
    benchmark_data['Model'].append(model_name)
    benchmark_data['mAP@50'].append(metrics['mAP50'])
    benchmark_data['mAP@50-95'].append(metrics['mAP50_95'])
    benchmark_data['FPS'].append(metrics.get('FPS', 'N/A'))
    benchmark_data['Source'].append('Literature')

for model_name, metrics in your_results.items():
    benchmark_data['Model'].append(f"{model_name} (Ours - Phase 2)")
    benchmark_data['mAP@50'].append(metrics['mAP50'])
    benchmark_data['mAP@50-95'].append(metrics['mAP50_95'])
    benchmark_data['FPS'].append(metrics['FPS'])
    benchmark_data['Source'].append('This Work')

benchmark_df = pd.DataFrame(benchmark_data)

print("Benchmarking Comparison:")
print("=" * 80)
print(benchmark_df.to_string(index=False))

# Visualization
if len(benchmark_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    lit_models = benchmark_df[benchmark_df['Source'] == 'Literature']['Model'].tolist()
    ours_models = benchmark_df[benchmark_df['Source'] == 'This Work']['Model'].tolist()
    
    # mAP@50 comparison
    axes[0].barh(['Literature Avg', 'Our Models Avg'], 
                 [benchmark_df[benchmark_df['Source'] == 'Literature']['mAP@50'].mean(),
                  benchmark_df[benchmark_df['Source'] == 'This Work']['mAP@50'].mean()],
                 color=['gray', 'blue'], alpha=0.7)
    axes[0].set_xlabel('mAP@50')
    axes[0].set_title('Average mAP@50: Literature vs Our Models')
    axes[0].grid(True, alpha=0.3)
    
    # mAP@50-95 comparison
    axes[1].barh(['Literature Avg', 'Our Models Avg'],
                 [benchmark_df[benchmark_df['Source'] == 'Literature']['mAP@50-95'].mean(),
                  benchmark_df[benchmark_df['Source'] == 'This Work']['mAP@50-95'].mean()],
                 color=['gray', 'blue'], alpha=0.7)
    axes[1].set_xlabel('mAP@50-95')
    axes[1].set_title('Average mAP@50-95: Literature vs Our Models')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()



## 9. Visual Comparison on Same Images

In [ ]:
# Visualize predictions from all models on the same validation images

def plot_all_predictions(models_dict, dataset, indices, score_thresh=0.5):
    """Plot predictions from all models side by side"""
    for idx in indices:
        img, gt = dataset[idx]
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = np.clip(img_np, 0, 1)
        
        fig, axes = plt.subplots(1, len(models_dict) + 1, figsize=(5 * (len(models_dict) + 1), 5))
        axes = axes.flatten()
        
        # Ground truth
        axes[0].imshow(img_np)
        axes[0].set_title('Ground Truth')
        axes[0].axis('off')
        ax_gt = axes[0]
        for box in gt['boxes']:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red', linewidth=2)
            ax_gt.add_patch(rect)
        
        # Model predictions
        for i, (model_name, model) in enumerate(models_dict.items(), 1):
            axes[i].imshow(img_np)
            axes[i].set_title(model_name)
            axes[i].axis('off')
            # Add prediction visualization code for each model type
            # (Implementation depends on model type)
        
        plt.tight_layout()
        plt.show()

# Select random validation images for comparison
sample_indices = random.sample(range(len(val_dataset_eval)), min(3, len(val_dataset_eval)))
print(f"Visualizing predictions on {len(sample_indices)} sample images...")

# Plot predictions (implementation needed for each model type)
print("\nVisual comparison implementation:")
print("  - Or implement unified visualization function above")

## 10. Discussion & Reflection

### Overall Findings:

**Best Performing Model:**
- Based on mAP@50 and mAP@50-95 metrics
- Consideration of inference speed (FPS)
- Trade-offs between accuracy and speed

### What Worked in Phase 2:

1. **Dataset Augmentation:**
   - Impact of larger augmented dataset on model performance
   - Improvements in generalization

2. **Model Selection:**
   - Which models performed best on this task
   - Architecture advantages for parking detection

3. **Training Strategy:**
   - Effective hyperparameter configurations
   - Training techniques that worked well

### Challenges Faced:

1. **Training Difficulties:**
   - [Document specific challenges]

2. **Dataset Issues:**
   - [Document any dataset-related challenges]

3. **Model Limitations:**
   - [Document limitations of each model]

### Dataset Impact Analysis:

- **Phase 1 vs Phase 2:** Performance improvements from augmented dataset
- **Dataset Size:** Impact of ~36k images vs smaller original dataset
- **Quality:** Effect of data augmentation on model robustness

### Recommendations:

**For Real-Time Deployment:**
- Best model for speed requirements
- Consideration of hardware constraints

**For Maximum Accuracy:**
- Best model when speed is not critical
- Use cases where accuracy is paramount

**For Balanced Performance:**
- Best overall model considering both speed and accuracy
- General deployment recommendations

### Future Work:

1. **Model Improvements:**
   - [Suggestions for improving models]

2. **Dataset Enhancements:**
   - [Suggestions for dataset improvements]

3. **Deployment Considerations:**
   - [Real-world deployment factors]

In [ ]:
# Save comprehensive comparison summary
summary_data = {
    'evaluation_results': evaluation_results,
    'fps_results': fps_results,
    'best_model_mAP50': max([r.get('mAP50', 0) for r in evaluation_results.values()]) if evaluation_results else 0,
    'best_model_mAP50_95': max([r.get('mAP50_95', 0) for r in evaluation_results.values()]) if evaluation_results else 0,
    'fastest_model': max(fps_results.items(), key=lambda x: x[1])[0] if fps_results else None,
}

summary_file = BASE_DIR / "checkpoints" / "phase2_comparison_summary.json"
summary_file.parent.mkdir(parents=True, exist_ok=True)

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\nComparison summary saved to: {summary_file}")

## Generate Model Outputs Comparison for Report

This cell generates the model outputs comparison visualization showing 2 original images + outputs from 3 best models (Faster R-CNN, DETR, SSD).


In [ ]:
# Generate Model Outputs Comparison Figure for Report
import matplotlib.patches as patches
from PIL import Image

# Helper functions for inference
def run_inference_frcnn_ssd(model, image_tensor, threshold=0.5):
    """Run inference for Faster R-CNN or SSD"""
    with torch.no_grad():
        predictions = model([image_tensor])[0]
    
    boxes = []
    labels = []
    scores = []
    
    for i in range(len(predictions['boxes'])):
        if predictions['scores'][i] > threshold:
            boxes.append(predictions['boxes'][i].cpu().numpy())
            labels.append(predictions['labels'][i].cpu().item())
            scores.append(predictions['scores'][i].cpu().item())
    
    return boxes, labels, scores

def run_inference_detr(model, image_tensor, processor, threshold=0.5):
    """Run inference for DETR"""
    image_pil = Image.fromarray((image_tensor.cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8))
    inputs = processor(images=image_pil, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    results = processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=torch.tensor([image_pil.size[::-1]])
    )[0]
    
    boxes = []
    labels = []
    scores = []
    
    for box, label, score in zip(results['boxes'], results['labels'], results['scores']):
        boxes.append(box.cpu().numpy())
        labels.append(label.item())
        scores.append(score.item())
    
    return boxes, labels, scores

# Class colors
class_colors = {"free_parking_space": "lime", "not_free_parking_space": "red"}

# Select 2 sample images
random.seed(42)
sample_indices = random.sample(range(len(val_dataset_eval)), min(2, len(val_dataset_eval)))
print(f"Selected sample indices: {sample_indices}")

# Create combined figure
fig_combined, axes_combined = plt.subplots(2, 4, figsize=(20, 10))

for img_idx, dataset_idx in enumerate(sample_indices):
    # Load image from dataset
    img, tgt = val_dataset_eval[dataset_idx]
    img_gpu = img.to(device)
    img_np = img.permute(1, 2, 0).numpy()
    if img_np.max() > 1:
        img_np = img_np / 255.0
    
    # Original image
    axes_combined[img_idx, 0].imshow(img_np)
    axes_combined[img_idx, 0].set_title("Original Image", fontsize=11, fontweight='bold')
    axes_combined[img_idx, 0].axis('off')
    
    # Run inference with each model
    model_names = ['Faster R-CNN', 'DETR', 'SSD']
    for model_idx, model_name in enumerate(model_names):
        ax = axes_combined[img_idx, model_idx + 1]
        
        if model_name not in models_loaded:
            ax.text(0.5, 0.5, f"{model_name}\nNot Available", 
                   ha='center', va='center', fontsize=10)
            ax.axis('off')
            continue
        
        model = models_loaded[model_name]
        
        try:
            if model_name == 'DETR':
                if processor is None:
                    processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
                boxes, labels, scores = run_inference_detr(model, img_gpu, processor, threshold=0.3)
            else:
                boxes, labels, scores = run_inference_frcnn_ssd(model, img_gpu, threshold=0.3)
            
            # Plot image with predictions
            ax.imshow(img_np)
            ax.set_title(f"{model_name}\n({len(boxes)} detections)", fontsize=11, fontweight='bold')
            ax.axis('off')
            
            # Draw bounding boxes
            for box, label, score in zip(boxes, labels, scores):
                x_min, y_min, x_max, y_max = box
                width = x_max - x_min
                height = y_max - y_min
                
                # Map label to class name (torchvision uses 1=free, 2=occupied)
                if label == 1:
                    class_name = "free_parking_space"
                elif label == 2:
                    class_name = "not_free_parking_space"
                else:
                    class_name = "unknown"
                
                color = class_colors.get(class_name, "yellow")
                rect = patches.Rectangle(
                    (x_min, y_min), width, height,
                    linewidth=1.5, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                
                # Add label with score for high confidence detections
                if score > 0.5:
                    ax.text(x_min, y_min - 3, f"{score:.2f}",
                           color=color, fontsize=7, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))
        except Exception as e:
            ax.text(0.5, 0.5, f"{model_name}\nError: {str(e)[:50]}", 
                   ha='center', va='center', fontsize=9)
            ax.axis('off')

plt.suptitle("Model Outputs Comparison: Original Images vs. Predictions from Best Models", 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.99])

# Save the combined figure
REPORT_OUTPUT_DIR = BASE_DIR / "report" / "temp"
REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = REPORT_OUTPUT_DIR / "model_outputs_comparison.png"
fig_combined.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved comparison figure to: {output_path}")

plt.show()
